In [ ]:
import sqlite3

import pandas as pd

conn = sqlite3.connect("../data/incentive-util.db")


def get_item_id_set():
    df = pd.read_sql_query("SELECT id FROM customers", conn)
    return set(df["id"])


def insert(to_insert: list):
    conn.executemany(
        """INSERT INTO customers (id, name)
        VALUES (:id, :name)""",
        to_insert,
    )
    conn.commit()


def update(to_update: list):
    conn.executemany(
        """UPDATE items
        SET name=:name
        WHERE id=:id""",
        to_update,
    )
    conn.commit()

In [2]:
def build_cleaned_list(import_path: str):
    df = pd.read_excel(import_path)
    df_clean = df.dropna(how="all").dropna(how="all", axis=1)
    ITEM_ID = 0
    ITEM_NAME = 1
    return [
        {
            "id": row[ITEM_ID],
            "name": row[ITEM_NAME]
        }
        for row in df_clean.itertuples(index=False)
    ]


def filter_to_insert(existing_ids: set, items: list):
    return [i for i in items if i["id"] not in existing_ids]

def filter_to_update(existing_ids: set, items: list):
    return [i for i in items if i["id"] in existing_ids]

In [4]:
import_path = "customers.xlsx"
items = build_cleaned_list(import_path)
existing_ids = get_item_id_set()
to_insert = filter_to_insert(existing_ids, items)
to_update = filter_to_update(existing_ids, items)

insert(to_insert)
update(to_update)
